# Small Files, Compaction, Layout, and Estimated Energy

This notebook summarizes recorded results for one local synthetic TPC-H `lineitem` sample. It compares the same query workload on the small-file baseline, compacted copy, sorted copy, and custom Morton-key ordered copy. It also reports the Issue #9 schema/provenance audit.

> Energy and electricity cost use an assumed average system wattage and an explicitly supplied tariff. They are estimates, not measured power. DuckDB has no native Z-order operation: the `z_ordered` label means the documented custom Morton ordering. The governance audit is a narrow schema/provenance control, not a privacy guarantee.

## Reproduce inputs
Run the documented pipeline scripts from the repository root before executing this notebook. Example paths are under the ignored `data/` directory:

```powershell
python scripts/generate_tpch.py --scale-factor 0.001 --output data/tpch-sf001-issue3
python scripts/ingest_small_files.py --input data/tpch-sf001-issue3/lineitem.parquet --output data/small-files-100rows-issue4 --batch-rows 100
python scripts/layout_variants.py --input data/small-files-100rows-issue4 --output-root data/layout-variants-issue7 --results data/layout-variants-issue7.json --file-size-bytes 262144 --warmups 1 --repeats 5 --threads 4
python scripts/estimate_energy_cost.py --input data/layout-variants-issue7.json --output data/energy-estimates-issue8.json --assumed-system-watts 50 --electricity-rate-inr-per-kwh 8
python scripts/governance_check.py --input data/small-files-100rows-issue4 --provenance data/tpch-sf001-issue3/manifest.json --output data/governance-issue9.json
```

The example wattage and tariff are scenario assumptions. Replace them with justified values for the report. Do not commit generated data or claim that these values are measured.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'scripts').is_dir() else cwd.parent
if not (PROJECT_ROOT / 'scripts').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks/ folder.')
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from analyze_results import analyze_files

# Select one complete run; never combine files from different directories.
data_root = PROJECT_ROOT / 'data'
layout_files = sorted(
    data_root.rglob('layout-variants-issue7.json'),
    key=lambda path: path.stat().st_mtime_ns,
    reverse=True,
)

RESULT_PATHS = None
for layout_path in layout_files:
    run_dir = layout_path.parent
    energy_path = run_dir / 'energy-estimates-issue8.json'
    audit_paths = (run_dir / 'governance-issue9.json', run_dir / 'governance-issue9-rerun.json')
    audit_path = next((path for path in audit_paths if path.is_file()), None)
    if energy_path.is_file() and audit_path is not None:
        RESULT_PATHS = {'layouts': layout_path, 'energy': energy_path, 'governance': audit_path}
        break
if RESULT_PATHS is None:
    raise FileNotFoundError(
        f'No complete results set found under {data_root}. Run the experiment first.'
    )

report = analyze_files(RESULT_PATHS['layouts'], RESULT_PATHS['energy'], RESULT_PATHS['governance'])
print('Inputs:')
for name, path in RESULT_PATHS.items():
    print(f'  {name}: {path.relative_to(PROJECT_ROOT)}')
print('Source metadata:', report['source_metadata'])

## Latency, result checks, and estimated energy

Each row comes from the matching raw layout and energy JSON. Speedup is baseline median divided by that layout's median; values below 1 indicate a slower result. The energy columns remain assumption-based.

In [ ]:
import pandas as pd
from IPython.display import display

query_table = pd.DataFrame(report['query_comparisons'])
display(query_table[['query', 'layout', 'run_count', 'median_seconds', 'min_seconds', 'max_seconds', 'speedup_vs_baseline', 'estimated_energy_wh', 'hypothetical_cost_inr', 'result_hash_match']])

In [ ]:
import matplotlib.pyplot as plt

layout_order = ['baseline', 'compacted', 'sorted', 'z_ordered']
latency = query_table.pivot(index='query', columns='layout', values='median_seconds').reindex(columns=layout_order)
ax = latency.plot(kind='bar', figsize=(10, 5), rot=0, ylabel='Median query latency (seconds)', title='Query latency by Parquet layout')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## File layout and one-time tuning overhead

File sizes and counts are measured values from the layout run. The Parquet writer's requested file size is approximate; use the recorded actual sizes. Optimization time is a one-time observed elapsed duration.

In [ ]:
storage_table = pd.DataFrame(report['storage_and_tuning'])
display(storage_table[['layout', 'file_count', 'total_bytes', 'min_bytes', 'max_bytes', 'mean_bytes', 'optimization_seconds']])
ax = storage_table.set_index('layout').reindex(layout_order)['file_count'].plot(kind='bar', figsize=(7, 4), rot=0, ylabel='Parquet file count', title='Files before and after layout tuning')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Energy assumptions and governance evidence

The estimator multiplies elapsed seconds by an assumed average whole-system wattage and divides by 3,600 to obtain Wh. Electricity cost is a hypothetical tariff calculation. The governance count describes only the allowlist/provenance checks.

In [ ]:
print('Energy assumptions:')
display(pd.Series(report['energy_assumptions'], name='value').to_frame())
energy_pivot = query_table.pivot(index='query', columns='layout', values='estimated_energy_wh').reindex(columns=layout_order)
ax = energy_pivot.plot(kind='bar', figsize=(10, 5), rot=0, ylabel='Estimated energy per query (Wh)', title='Estimated query energy (assumption-based)')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()
cost_pivot = query_table.pivot(index='query', columns='layout', values='hypothetical_cost_inr').reindex(columns=layout_order)
ax = cost_pivot.plot(kind='bar', figsize=(10, 5), rot=0, ylabel='Hypothetical electricity cost per query (INR)', title='Hypothetical electricity cost (assumption-based)')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()
print('Governance audit metrics:')
display(pd.Series(report['governance'], name='value').to_frame())

## Interpretation and limitations

Use the tables and figures alongside the three input JSON files. Report the local platform, DuckDB version, thread count, warmups, measured repeats, and cache policy from the source metadata. Avoid extrapolating this sample's results to a cluster or millions of files. A measured latency gain does not establish an energy gain unless the stated power assumption and query workload are considered. A passing schema audit does not establish that values are non-personal or quantify re-identification risk.

The custom Morton layout is only an experiment over date and return-flag values; it is not a native DuckDB Z-order feature. Include negative or mixed per-query outcomes and the one-time optimization cost in the report.